In [10]:
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
from datetime import date
import re

WRITE_FILE_PATH = r'c:\Interest_rate_files'
URL = "https://www.capitalbank.co.in/interest-rates/callable-domestic-term-deposit"
TAX_SAVER_URL = url="https://www.capitalbank.co.in/interest-rates/tax-saver-fixed-deposit"

date1 = (date.today()).strftime('%d%m%Y') # Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' # Set directory path to write file

r=requests.get(URL)
# print(r)
soup=BeautifulSoup(r.text,"html.parser")
table=soup.find_all("table",class_="table")
table1=table[0]
table2=table[1]
rows=table1.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

rates = []
for i in rows[1:]:  # Skip heading
    data=i.find_all("td")
    #print(data)
    row=[tr.text for tr in data]
    rates.append(row)

rates = [item for item in rates if item != ['Special category']]
#print(rates)
##########

rows=table2.find_all("tr")
senior_rates = []
for i in rows[1:]:  # Skip heading
    data=i.find_all("td")
    #print(data)
    row=[tr.text for tr in data]
    senior_rates.append(row)
#print(senior_rates)

for rate in rates:
    period = rate[0]
    # Find the matching senior rate
    senior_rate_found = False
    for senior_rate in senior_rates:
        if senior_rate[0] == period:
            rate.append(senior_rate[1])
            senior_rate_found = True
            break
    if not senior_rate_found:
        rate.append('NA')  # Append 'NA' if no matching period is found in SENIOR_RATES

############################TAX SAVER RATES
r=requests.get(TAX_SAVER_URL)
soup=BeautifulSoup(r.text,"html.parser")
table=soup.find("table",class_="table")
rows=table.find_all("tr")

tax_saver=[]
for i in rows[1:2]:  # Skip heading
    data=i.find_all("td")
    #print(data)
    row=[tr.text for tr in data]
    row[0] = 'Tax-saver Fixed Deposit'
    tax_saver.append(row)

seniorrate=float(tax_saver[0][1][:4])+0.50
seniorrate=str(seniorrate) + '%'
tax_saver[0].append(seniorrate)

rates.append(tax_saver[0])
#print(rates)

final_rates = []
for i in rates:
    x=['Capital Small Finance Bank']
    x.extend(i)
    x.insert(3, '')
    x.extend([''])
    final_rates.append(x)
#print(final_rates)
    
pd.set_option('display.max_columns', None)
int_rate_df=pd.DataFrame(final_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate','sr_rate',
                                        'annualised_sr_rate'])
#print(int_rate_df)
int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
print("\nCapital SF Bank = Success. Number of rows added = ", len(final_rates))

Today's date = 19112024
Interest rate file name =  c:\Interest_rate_files\Interest_rate_v2_19112024.csv
[['Capital Small Finance Bank', '15 Days to 30 Days', '3.50%', '', '4.00%', ''], ['Capital Small Finance Bank', '31 Days to 45 Days', '3.50%', '', '4.00%', ''], ['Capital Small Finance Bank', '46 Days to 89 Days', '4.00%', '', '4.50%', ''], ['Capital Small Finance Bank', '90 Days to 179 Days', '4.75%', '', '5.25%', ''], ['Capital Small Finance Bank', '180 Days to less than 1 Year', '5.50%', '', '6.00%', ''], ['Capital Small Finance Bank', '1 Year to less than 5 Years', '7.15%', '', '7.65%', ''], ['Capital Small Finance Bank', '5 Years and upto 10 Years', '7.10%', '', '7.60%', ''], ['Capital Small Finance Bank', '12 Months', '7.50%', '', '8.00%', ''], ['Capital Small Finance Bank', '400 Days', '7.55%', '', '8.05%', ''], ['Capital Small Finance Bank', '600 Days', '7.40%', '', '7.90%', ''], ['Capital Small Finance Bank', '900 Days', '7.30%', '', '7.80%', ''], ['Capital Small Finance Ban